# SparkSQL Window Functions · Olist demosu

*VBO Data Engineering Bootcamp · 8. hafta*

> **İş sorusu:** Müşteri ikinci kez geliyor mu, ne zaman geliyor? Her kategoride hangi satıcılar öne çıkıyor?

Sunumda yöntemleri gördük. Burada tek bir veri setinde **doğru yöntemi seçerek** beş soruyu cevaplıyoruz. Her perdenin ritmi aynı:

1. **İş sorusu**
2. Hangi fonksiyon, `PARTITION BY` ne, `ORDER BY` ne?
3. Sorgu
4. **Sağlama:** sonucu bağımsız bir yoldan doğrulayan küçük kontrol (`assert`)

| Perde | Soru | Fonksiyon |
|---|---|---|
| 1 | Müşterinin kaçıncı siparişi? | ? |
| 2 | İki sipariş arası kaç gün? | ? |
| 3 | Aylık ciro: running total, 3 aylık ortalama | ? |
| 4 | Kategori içinde ilk 3 satıcı | ? |
| 5 | Harcamaya göre müşteri çeyrekleri | ? |

## 0. SparkSession ve Olist view'ları

CSV'ler `notebooks/datasets/olist/` altında olmalı (indirme adımı README'de). Her CSV bir temp view olur; bundan sonrası yalnızca `spark.sql`.

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Window Functions · Olist")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")   # varsayılan 200; tek makinede küçük veri için 8 yeter
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")   # WARN logları hücre çıktısını kirletmesin

DATA = Path("datasets/olist")
VIEWS = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
    "payments": "olist_order_payments_dataset.csv",
}
missing = [f for f in VIEWS.values() if not (DATA / f).exists()]
assert not missing, f"Eksik CSV: {missing}. README > Veriyi indir adımını uygula."

for view, file in VIEWS.items():
    spark.read.csv(str(DATA / file), header=True, inferSchema=True).createOrReplaceTempView(view)

spark.sql("SHOW VIEWS").show()

## Perde 1 · Müşterinin kaçıncı siparişi?

**İş sorusu:** Her siparişe "bu, müşterinin kaçıncı siparişi?" numarasını verelim. Kaç müşteri ikinci kez geliyor?

**Sohbete sor:** Hangi fonksiyon? `PARTITION BY` ne? `ORDER BY` ne?  ⏸ *Bekle, cevapları oku.*

### Spark UI'da bak

Window sorgusu Spark'ta iki operatöre dönüşür: **`Exchange hashpartitioning(customer_unique_id)`** (aynı müşterinin satırları aynı partition'a taşınır, yani shuffle) ve **`Window`**. Canlıda: http://localhost:4040 → **SQL / DataFrame** sekmesi → son sorgu. Aşağıdaki hücre aynı planı metin olarak gösterir.

## Perde 2 · İki sipariş arası kaç gün?

**İş sorusu:** Tekrar gelen müşteri, bir önceki siparişinden kaç gün sonra geliyor?

**Sohbete sor:** Hangi fonksiyon? `PARTITION BY` ne? `ORDER BY` ne?  ⏸ *Bekle, cevapları oku.*

## Perde 3 · Aylık ciro: running total ve 3 aylık hareketli ortalama

**İş sorusu:** Teslim edilen siparişlerde aylık ciro nasıl birikiyor, kısa vadeli eğilim ne?

Tam aylar: 2017-01 … 2018-08. Ciro = `payments.payment_value`.

**Sohbete sor:** Hangi fonksiyon? `PARTITION BY` ne? `ORDER BY` ne?  ⏸ *Bekle, cevapları oku.* *Ve frame ne olmalı?*

**`ROWS` mu `RANGE` mı?** `ROWS BETWEEN 2 PRECEDING` son 3 *satırı* alır, `RANGE BETWEEN INTERVAL 2 MONTHS PRECEDING` son 3 *takvim ayını*: bir ay eksik olsaydı (Olist'te 2016-11 yok) `ROWS` penceresi 4 aya yayılırdı. Burada her ay dolu, ikisi aynı sonucu verir.

**`PARTITION BY` yok:** tüm aylar tek sıralamada, Spark bütün veriyi **tek partition'a** toplar (log: `No Partition Defined for Window operation!`). 20 satırda sorun değil; milyarlık tabloda darboğaz.

## Perde 4 · Kategori içinde ilk 3 satıcı

**İş sorusu:** Her kategoride ciroya göre öne çıkan ilk 3 satıcı kim?

**Sohbete sor:** Hangi fonksiyon? `PARTITION BY` ne? `ORDER BY` ne?  ⏸ *Bekle, cevapları oku.*

**Not:** Gerçek veride ilk 3 içinde birebir eşit ciro **yok**. Raporda ciroyu **yüz BRL'ye yuvarlıyoruz** (`ROUND(revenue, -2)`); yönetim "9.100 BRL" ile "9.149 BRL"yi zaten aynı görür. Eşitlik bu yuvarlamadan doğuyor, bunu bilerek yapıyoruz.

### `QUALIFY` Spark'ta var mı?

Databricks, Snowflake, BigQuery gibi motorlarda window sonucu alt sorgusuz filtrelenir: `... QUALIFY rnk <= 3`. **Açık kaynak Spark 4.1.3'te yok**, `PARSE_SYNTAX_ERROR` verir. Bu yüzden yukarıda view/alt sorgu + `WHERE` kullandık.

## Perde 5 · Harcamaya göre müşteri çeyrekleri

**İş sorusu:** Müşterileri toplam harcamaya göre dört eşit gruba bölelim. En üst çeyrek cironun ne kadarını getiriyor?

**Sohbete sor:** Hangi fonksiyon? `PARTITION BY` ne? `ORDER BY` ne?  ⏸ *Bekle, cevapları oku.*

## Kapanış · İş sorusuna cevap

> *Müşteri ikinci kez geliyor mu, ne zaman geliyor? Her kategoride hangi satıcılar öne çıkıyor?*

| Perde | Yöntem | Cevap |
|---|---|---|
| 1 | `ROW_NUMBER`, `PARTITION BY customer_unique_id` | Müşterilerin ~%3'ü ikinci kez geliyor. Partition'ı yanlış anahtarla kurunca herkes "ilk sipariş" görünüyordu. |
| 2 | `LAG` | Tekrar siparişlerin %29'u aynı gün; gerçek dönüşte medyan 67 gün. Hatırlatma penceresi 1-2. ay. |
| 3 | `SUM/AVG OVER` + `ROWS BETWEEN` | 2017'de büyüme, Kasım 2017 tepe, 2018'de ~1 milyon BRL/ay yatay. |
| 4 | `DENSE_RANK` | Her kategorinin ilk 3 satıcısı; `ROW_NUMBER` yüz BRL'ye yuvarlanmış ciroda 13 satıcıyı sessizce düşürüyordu. QUALIFY Spark 4.1.3'te yok. |
| 5 | `NTILE(4)` | En üst çeyrek cironun ~%59'u, en alt çeyrek ~%7'si. |

**Yöntem seçerken sorulacak üç soru:** Satır başına ne üretiyorum (sıra, önceki değer, birikim, grup)? Kimin içinde (`PARTITION BY`)? Hangi sırayla ve hangi pencerede (`ORDER BY`, frame)?